In [23]:
import psycopg2
import random
import csv

conn = psycopg2.connect(
    dbname="ava_um_db",
    user="ava_um_usr",
    password="ava_um_pswd",
    host="localhost",
    port="5432"
)

cursor = conn.cursor()

CGH_data = 'Centrais_Geradoras_Hidrelétricas_CGH.csv'
UHE_data = 'Usinas_Hidrelétricas_UHE.csv'

In [24]:
try:
    def criar_tabelas(cursor):
        tabelas_ddl = """/* ava_um_modeloLogico: */
        BEGIN;

        CREATE SCHEMA IF NOT EXISTS public AUTHORIZATION ava_um_usr;
        SET search_path TO public, public;

        DROP TABLE IF EXISTS public."Usina" CASCADE;
        DROP TABLE IF EXISTS public."Rios" CASCADE;
        DROP TABLE IF EXISTS public."Proprietarios" CASCADE;
        DROP TABLE IF EXISTS public."Municipios" CASCADE;
        DROP TABLE IF EXISTS public."Estados" CASCADE;
        DROP TABLE IF EXISTS public."Localizacoes" CASCADE;

        CREATE TABLE IF NOT EXISTS public."Usina" (
            id SERIAL PRIMARY KEY,
            nome VARCHAR NOT NULL,
            potkw FLOAT,
            potfis FLOAT,
            iniciop DATE,
            atualizacao DATE,
            lat DOUBLE PRECISION ,
            long DOUBLE PRECISION ,
            estagio VARCHAR,
            id_proprietario INTEGER NULL,
            id_rio INTEGER NOT NULL,
            id_localizacao INTEGER NOT NULL
        );

        CREATE TABLE IF NOT EXISTS public."Proprietarios" (
            proprietario VARCHAR,
            id_proprietario INTEGER PRIMARY KEY NOT NULL
        );

        CREATE TABLE IF NOT EXISTS public."Municipios" (
            municipio VARCHAR NOT NULL,
            uf CHAR(2) NOT NULL,
            id_municipio SERIAL PRIMARY KEY
        );

        CREATE TABLE IF NOT EXISTS public."Estados" (
            uf CHAR(2) PRIMARY KEY
        );

        CREATE TABLE IF NOT EXISTS public."Rios" (
            rio VARCHAR NOT NULL,
            id_rio INTEGER PRIMARY KEY
        );

        CREATE TABLE IF NOT EXISTS public."Localizacoes" (
            id_localização INTEGER PRIMARY KEY,
            longitude DOUBLE PRECISION,
            id_municipio1 INTEGER NOT NULL,
            id_municipio2 INTEGER,
            latitude DOUBLE PRECISION
        );

        ALTER TABLE IF EXISTS public."Usina"
        ADD CONSTRAINT FK_Usina_Proprietario
        FOREIGN KEY (id_proprietario)
        REFERENCES public."Proprietarios"(id_proprietario);

        ALTER TABLE IF EXISTS public."Usina"
        ADD CONSTRAINT FK_Usina_Rio
        FOREIGN KEY (id_rio)
        REFERENCES public."Rios"(id_rio);

        ALTER TABLE IF EXISTS public."Usina"
        ADD CONSTRAINT FK_Usina_Localizacao
        FOREIGN KEY (id_localizacao)
        REFERENCES public."Localizacoes"(id_localização);

        ALTER TABLE IF EXISTS public."Municipios" ADD CONSTRAINT FK_Municipios_1
            FOREIGN KEY (uf)
            REFERENCES public."Estados"(uf);
        
        ALTER TABLE IF EXISTS public."Localizacoes"
        DROP CONSTRAINT IF EXISTS FK_Localizacoes_2;

        ALTER TABLE IF EXISTS public."Localizacoes"
        ADD CONSTRAINT FK_Localizacoes_Mun1
            FOREIGN KEY (id_municipio1)
            REFERENCES public."Municipios"(id_municipio);

        ALTER TABLE IF EXISTS public."Localizacoes"
        ADD CONSTRAINT FK_Localizacoes_Mun2
            FOREIGN KEY (id_municipio2)
            REFERENCES public."Municipios"(id_municipio);
        COMMIT;"""
        cursor.execute(tabelas_ddl)
        conn.commit()
except psycopg2.Error as e:
    print("Falha ao criar schema:  {e}")
    conn.rollback()

In [25]:

def carregar_dados(cursor, filepath):
    counter_IDusina = 0
    id_rio = 0
    id_prop = 0
    id_local = 0
    id_mun1 = 0
    id_mun2 = 0
    with open(filepath, 'r', encoding='utf-8') as csvfile:
        leitor = csv.reader(csvfile, delimiter=';')
        header = next(leitor)

        for linha in leitor:
            nome_usina = linha[0]
            nome_proprietario = linha[1]
            mun_1 = linha[2]
            uf_1 = linha[3]
            mun_2 = linha[4]
            uf_2 = linha[5]
            nome_rio = linha[6]
            potkw = linha[7].replace(',', '.')
            atualizacao = linha[8]
            if atualizacao == "00000000":
                atualizacao = None
            iniop = linha[9]
            if iniop == "00000000":
                iniop = None
            potfis = linha[10].replace(',', '.')
            id_usina = linha[11]
            if id_usina == '0':
                id_usina = random.random() + counter_IDusina + 1
                cursor.execute('SELECT id FROM public."Usina"')
                ids_usados = [row[0] for row in cursor.fetchall()]
                while id_usina in ids_usados:
                    id_usina = random.random() + counter_IDusina + 1
            lat = linha[12].replace(',', '.')
            long = linha[13].replace(',', '.')
            estag_imp = linha[14]

            cursor.execute(
                """
                INSERT INTO public."Estados" (uf)
                VALUES (%s)
                ON CONFLICT (uf) DO NOTHING;
                """,
                (uf_1,)
            )
            if uf_2:
                cursor.execute(
                    """
                    INSERT INTO public."Estados" (uf)
                    VALUES (%s)
                    ON CONFLICT (uf) DO NOTHING;
                    """,
                    (uf_2,)
                )
            
            if id_local == 0:
                cursor.execute('SELECT COALESCE(MAX(id_localização), 0) FROM public."Localizacoes"')
                ultimo_id = cursor.fetchone()[0]
                id_local = ultimo_id + 1
            else:
                id_local+=1
            
            cursor.execute(
                """
                    INSERT INTO public."Municipios" (municipio, uf)
                    VALUES (%s, %s)
                    RETURNING id_municipio
                    """,
                    (mun_1, uf_1)
            )
            result = cursor.fetchone()
            if result:
                id_mun1 = result[0]
            
            if mun_2:
                cursor.execute(
                    """
                    INSERT INTO public."Municipios" (municipio, uf)
                    VALUES (%s, %s)
                    RETURNING id_municipio
                    """,
                    (mun_2, uf_2)
                )
                result = cursor.fetchone()
                if result:
                    id_mun2 = result[0]
                cursor.execute(
                    """
                    INSERT INTO public."Localizacoes" (id_localização, longitude, id_municipio1, id_municipio2, latitude)
                    VALUES (%s, %s, %s, %s, %s)
                    """,
                    (id_local, long, id_mun1, id_mun2, lat)
                )
            else:
                cursor.execute(
                    """
                    INSERT INTO public."Localizacoes" (id_localização, longitude, id_municipio1, id_municipio2, latitude)
                    VALUES (%s, %s, %s, %s, %s)
                    """,
                    (id_local, long, id_mun1, None, lat)
                )

            if nome_proprietario:
                if id_prop == 0:
                    cursor.execute('SELECT COALESCE(MAX(id_proprietario), 0) FROM public."Proprietarios"')
                    ultimo_id = cursor.fetchone()[0]
                    id_prop = ultimo_id + 1
                else:
                    id_prop +=1
                cursor.execute(
                """
                    INSERT INTO public."Proprietarios" (proprietario, id_proprietario)
                    VALUES (%s, %s)
                    """,
                    (nome_proprietario, id_prop)
                )
            
            if id_rio == 0:
                cursor.execute('SELECT COALESCE(MAX(id_rio), 0) FROM public."Rios"')
                ultimo_id = cursor.fetchone()[0]
                id_rio = ultimo_id + 1
            else:
                id_rio+=1
            cursor.execute(
                """
                    INSERT INTO public."Rios" (rio, id_rio)
                    VALUES (%s, %s)
                    """,
                    (nome_rio, id_rio)
                )
            if id_prop == 0:
                id_prop = None
            cursor.execute(
                """
                INSERT INTO public."Usina" (nome, potkw, potfis, iniciop, atualizacao, lat, long, estagio, id_proprietario, id_rio, id_localizacao)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)
                """,
                (nome_usina, potkw, potfis, iniop, atualizacao, lat, long, estag_imp, id_prop, id_rio, id_local)
            )
            id_prop = 0
    try:
        conn.commit()
    except psycopg2.Error as e:
        print("Ocorreu um erro: {e}")
        conn.rollback()

In [26]:
def fazer_query(sql_query):
    try:
        conn = psycopg2.connect(
            dbname="ava_um_db", user="ava_um_usr", password="ava_um_pswd", host="localhost", port="5432"
        )
        cursor = conn.cursor()

        cursor.execute(sql_query)
        #colnames = [desc[0] for desc in cursor.description]
        results = cursor.fetchall()
        print(results)
    except Exception as e:
        print("Ocorreu um erro: {e}")

In [27]:
criar_tabelas(cursor)

In [28]:
carregar_dados(cursor, CGH_data)
carregar_dados(cursor, UHE_data)
cursor.close()
conn.close()

In [29]:
query_1 = """SELECT 
    p.proprietario,
    SUM(u.potfis) AS total_potfis
FROM public."Usina" u
JOIN public."Proprietarios" p ON u.id_proprietario = p.id_proprietario
GROUP BY p.proprietario
ORDER BY total_potfis DESC
LIMIT 3;"""

query_2 = """SELECT 
    uf,
    avg_potkw,
    RANK() OVER (ORDER BY avg_potkw DESC) AS rank_estado
FROM (
    SELECT 
        e.uf,
        AVG(u.potkw) AS avg_potkw
    FROM public."Usina" u
    JOIN public."Localizacoes" l ON u.id_localizacao = l.id_localização
    JOIN public."Municipios" m ON l.id_municipio1 = m.id_municipio
    JOIN public."Estados" e ON m.uf = e.uf
    GROUP BY e.uf
) sub;
"""

query_3 = """SELECT 
    u.id,
    u.nome,
    u.id_proprietario
FROM public."Usina" u
WHERE u.id_proprietario IS NULL
   OR NOT EXISTS (
       SELECT 1
       FROM public."Proprietarios" p
       WHERE p.id_proprietario = u.id_proprietario
   );
"""

query_4 = """WITH RECURSIVE cadeia_usinas AS (
    -- base: usina inicial (por exemplo, id = 1)
    SELECT id, nome, id_rio
    FROM public."Usina"
    WHERE id = 1

    UNION ALL

    -- recursão: usinas que compartilham o rio da usina anterior
    SELECT u.id, u.nome, u.id_rio
    FROM public."Usina" u
    JOIN cadeia_usinas cu ON u.id_rio = cu.id_rio
    WHERE u.id <> cu.id
)
SELECT * FROM cadeia_usinas;
"""

query_5 = """SELECT
    id,
    nome,
    lat,
    long,
    6371 * 2 * 
      ATAN2(
        SQRT(
          SIN(RADIANS(u.lat - :lat0) / 2)^2 +
          COS(RADIANS(:lat0)) * COS(RADIANS(u.lat)) *
          SIN(RADIANS(u.long - :long0) / 2)^2
        ),
        SQRT(1 - (
          SIN(RADIANS(u.lat - :lat0) / 2)^2 +
          COS(RADIANS(:lat0)) * COS(RADIANS(u.lat)) *
          SIN(RADIANS(u.long - :long0) / 2)^2
        ))
      ) AS distancia_km
FROM public."Usina" u
ORDER BY distancia_km
LIMIT 5;
"""

fazer_query(query_1)
fazer_query(query_2)
fazer_query(query_3)
fazer_query(query_4)
fazer_query(query_5)

[('Rio Paraná Energia S.A.', 4995200.0), ('Rio Paranapanema Energia S.A.', 1682956.0), ('Companhia Energética de São Paulo', 1654620.0)]
[('MS', 1540000.0, 1), ('MG', 507670.0, 2), ('PR', 237360.0, 3), ('SP', 113926.24084745762, 4)]
[(1, 'Arroio das Flores', None), (2, 'Ribeirão da Lagoa', None), (5, 'Catas Altas III', None), (24, 'Quilombo I', None), (37, 'Funil', None), (47, 'Eleutério (rio Euletério)', None), (116, 'Viradouro', None), (118, 'Banharão', None), (119, 'Salto do Itapura', None), (120, 'Barretos', None)]
[(1, 'Arroio das Flores', 1)]
Ocorreu um erro: {e}
